In [1]:
from pathlib import Path

LAST_NAME = "PARRAS"
STUDENT_ID = "TUPM-26-3597"

SEED_DIGIT = int(STUDENT_ID[-1])
ID_SUM = sum(int(d) for d in STUDENT_ID if d.isdigit())
NAME_LENGHT =len(LAST_NAME)

print("Seed:", SEED_DIGIT)
print("ID sum:", ID_SUM)
print("Name Lenght:", NAME_LENGHT)


Seed: 7
ID sum: 32
Name Lenght: 6


In [ ]:
import csv
import json
import shutil
from collections import Counter
from pathlib import Path

In [28]:

# Import tools for reading, writing, copying, and counting data.
import csv
import json
import shutil
from pathlib import Path
from collections import Counter

# Find the folder where this notebook is saved.
folder = Path.cwd()

# Tell Python where the original CSV file is located.
input_file = folder / "weblog.csv"



# Calculate how many records to process.
records_to_process = 1000 + ID_SUM

# Calculate the target HTTP status class.
target_class = (SEED_DIGIT % 4) + 2

# Prepare the names of the output files.
filtered_file = folder / "filtered_logs.csv"
summary_file = folder / "summary.json"
backup_file = folder / "summary_backup.json"

# Display the setup values.
print("Records to process:", records_to_process)
print("Target HTTP status class:", str(target_class) + "xx")
print("Input file:", input_file)


Records to process: 1000
Target HTTP status class: 2xx
Input file: c:\Users\JULIANNA\OneDrive\Desktop\VISUAL CODE\Lab6_File_HAndling_Parras\weblog.csv


In [29]:

input_file = Path(r"C:\Users\JULIANNA\Downloads\weblog.csv\weblog.csv")

# Open the CSV file for reading.
with open(input_file, "r", newline="", encoding="utf-8-sig") as csv_file:

    # Read one row at a time as a dictionary.
    reader = csv.DictReader(csv_file)

    # Get the column names from the CSV header.
    headers = reader.fieldnames

    # Remove extra spaces from the column names.
    headers = [name.strip() for name in headers]

    # Find the actual status column name.
    status_column = next(
        (name for name in headers if name.lower() in ("status", "staus")),
        None
    )

    # Make sure all required columns exist.
    required_columns = ["IP", "Time", "URL"]

    # Stop the program if a required column is missing.
    if status_column is None or not all(
        name in headers for name in required_columns
    ):
        raise ValueError("Required CSV columns are missing.")

    # Display the column names found in the dataset.
    print("CSV columns:", headers)
    print("Status column used:", status_column)


CSV columns: ['IP', 'Time', 'URL', 'Staus']
Status column used: Staus


In [24]:

# Create counters and lists for the assessment results.
status_counts = Counter()
url_counts = Counter()
records_in_target_class = 0
records_processed = 0

# Open the original CSV and the filtered CSV.
with open(input_file, "r", newline="", encoding="utf-8-sig") as csv_file, \
     open(filtered_file, "w", newline="", encoding="utf-8") as output_file:

    # Read the original records one at a time.
    reader = csv.DictReader(csv_file)

    # Write the same column names into the filtered CSV.
    writer = csv.DictWriter(output_file, fieldnames=headers)
    writer.writeheader()

    # Process each record without loading the entire dataset.
    for row in reader:

        # Stop after the required number of records.
        if records_processed >= records_to_process:
            break

        # Count this record as processed.
        records_processed += 1

        # Read the HTTP status code from the correct column.
        status = row[status_column].strip()

        # Ignore records with invalid or missing status codes.
        if len(status) != 3 or not status.isdigit():
            continue

        # Count how many times each HTTP status code appears.
        status_counts[status] += 1

        # Check whether the status belongs to the target class.
        if status.startswith(str(target_class)):

            # Count the matching record.
            records_in_target_class += 1

            # Count requests for this URL.
            url_counts[row["URL"].strip()] += 1

            # Save the matching record into the filtered CSV.
            writer.writerow(row)

# Get the five most frequently requested URLs.
top_five_urls = url_counts.most_common(5)

# Display the processing results.
print("Records processed:", records_processed)
print("Target status class:", str(target_class) + "xx")
print("Records in target class:", records_in_target_class)

# Display the count of each HTTP status code.
print("\nCount per status code:")
for status, count in sorted(status_counts.items()):
    print(status, ":", count)

# Display the five most requested URLs.
print("\nFive most requested URLs:")
for url, count in top_five_urls:
    print(url, ":", count)

# Display the filtered CSV location.
print("\nFiltered CSV saved to:", filtered_file)


Records processed: 1000
Target status class: 2xx
Records in target class: 831

Count per status code:
200 : 823
206 : 8
302 : 143
304 : 21
404 : 5

Five most requested URLs:
GET /login.php HTTP/1.1 : 104
GET /home.php HTTP/1.1 : 91
GET /css/normalize.css HTTP/1.1 : 38
GET /css/bootstrap.min.css HTTP/1.1 : 37
GET /js/vendor/jquery-1.12.0.min.js HTTP/1.1 : 37

Filtered CSV saved to: c:\Users\JULIANNA\OneDrive\Desktop\VISUAL CODE\Lab6_File_HAndling_Parras\filtered_logs.csv


In [30]:

# Prepare the summary using the results from Cell 3.
summary = {
    "records_processed": records_processed,
    "target_status_class": str(target_class) + "xx",
    "records_in_target_class": records_in_target_class,
    "status_counts": dict(status_counts),
    "top_five_urls": [
        {"url": url, "count": count}
        for url, count in top_five_urls
    ],
    "filtered_csv": filtered_file.name
}

# Write the summary into a JSON file.
with open(summary_file, "w", encoding="utf-8") as json_file:
    json.dump(summary, json_file, indent=4)

# Display the JSON summary file location.
print("JSON summary saved to:", summary_file)

# Display the actual contents of the JSON summary.
print("\nJSON summary results:")
print(json.dumps(summary, indent=4))


JSON summary saved to: c:\Users\JULIANNA\OneDrive\Desktop\VISUAL CODE\Lab6_File_HAndling_Parras\summary.json

JSON summary results:
{
    "records_processed": 1000,
    "target_status_class": "2xx",
    "records_in_target_class": 831,
    "status_counts": {
        "200": 823,
        "302": 143,
        "304": 21,
        "206": 8,
        "404": 5
    },
    "top_five_urls": [
        {
            "url": "GET /login.php HTTP/1.1",
            "count": 104
        },
        {
            "url": "GET /home.php HTTP/1.1",
            "count": 91
        },
        {
            "url": "GET /css/normalize.css HTTP/1.1",
            "count": 38
        },
        {
            "url": "GET /css/bootstrap.min.css HTTP/1.1",
            "count": 37
        },
        {
            "url": "GET /js/vendor/jquery-1.12.0.min.js HTTP/1.1",
            "count": 37
        }
    ],
    "filtered_csv": "filtered_logs.csv"
}


In [31]:

# Copy the JSON summary into a backup file.
shutil.copy2(summary_file, backup_file)

# Read the original JSON summary.
with open(summary_file, "r", encoding="utf-8") as original_file:
    original_data = json.load(original_file)

# Read the backup JSON file.
with open(backup_file, "r", encoding="utf-8") as backup:
    backup_data = json.load(backup)

# Check whether the original and backup contain the same data.
backup_verified = original_data == backup_data

# Display the backup verification result.
print("Backup file:", backup_file)
print("Backup verification:", backup_verified)


Backup file: c:\Users\JULIANNA\OneDrive\Desktop\VISUAL CODE\Lab6_File_HAndling_Parras\summary_backup.json
Backup verification: True


In [27]:

# Display all required assessment results.
print("========== ASSESSMENT DATA ==========")

print("Records processed:", records_processed)

print("Target status class:", str(target_class) + "xx")

print("Records in target class:", records_in_target_class)

print("\nCount per status code:")
for status, count in sorted(status_counts.items()):
    print(status, ":", count)

print("\nFive most requested URLs:")
for url, count in top_five_urls:
    print(url, ":", count)

print("\nFiltered CSV file:", filtered_file.name)

print("JSON summary file:", summary_file.name)

print("Backup verification:", backup_verified)

print("=====================================")


========== ASSESSMENT DATA ==========
Records processed: 1000
Target status class: 2xx
Records in target class: 831

Count per status code:
200 : 823
206 : 8
302 : 143
304 : 21
404 : 5

Five most requested URLs:
GET /login.php HTTP/1.1 : 104
GET /home.php HTTP/1.1 : 91
GET /css/normalize.css HTTP/1.1 : 38
GET /css/bootstrap.min.css HTTP/1.1 : 37
GET /js/vendor/jquery-1.12.0.min.js HTTP/1.1 : 37

Filtered CSV file: filtered_logs.csv
JSON summary file: summary.json
Backup verification: True
